# HAUS Rosalie — Rent Collection Dashboard

Drop in a **Multi-Tenant Ledger CSV from Rentec** export and run all cells.

**Outputs:**
1. **Heatmap** (shareable) — anonymized tenant × month grid, color-coded by end-of-month balance
2. **Private key** (treasurer only) — maps emoji IDs back to real names

Emoji assignments are re-shuffled every time you run the notebook.

In [ ]:
# ── CONFIG ──────────────────────────────────────────────
CSV_PATH = r"PATH_TO_CSV_HERE"

# One emoji per tenant — add/remove if tenant count changes.
EMOJI_POOL = ['\U0001F6F6', '\U0001F52A', '\U0001F3E1', '\U0001F920',
              '\U0001F415', '\U0001F408', '\U0001F496', '\U0001F9B6',
              '\U0001F333', '\U0001F40E']
# canoe, knife, house_with_garden, cowboy, dog, cat,
# sparkling_heart, foot, deciduous_tree, horse
# ────────────────────────────────────────────────────────

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from matplotlib.lines import Line2D
from PIL import Image, ImageDraw, ImageFont
import platform, os, glob, random

# ── BATLOW COLORMAP (Crameri, embedded) ────────────────
_batlow_anchors = [
    (0.00, (0.0104, 0.0302, 0.2082)),
    (0.11, (0.0780, 0.1439, 0.3523)),
    (0.22, (0.1510, 0.2268, 0.4220)),
    (0.33, (0.2472, 0.2977, 0.4185)),
    (0.44, (0.3658, 0.3534, 0.3878)),
    (0.55, (0.4900, 0.3960, 0.3400)),
    (0.66, (0.6190, 0.4340, 0.2700)),
    (0.77, (0.7500, 0.4890, 0.1600)),
    (0.88, (0.8700, 0.5950, 0.0500)),
    (1.00, (0.9800, 0.7500, 0.0500)),
]
batlow = mcolors.LinearSegmentedColormap.from_list(
    'batlow', [(p, c) for p, c in _batlow_anchors], N=256
)

# ── EMOJI FONT (cross-platform) ───────────────────────
def _find_emoji_font():
    system = platform.system()
    candidates = []
    if system == 'Windows':
        windir = os.environ.get('WINDIR', r'C:\\Windows')
        candidates = [
            os.path.join(windir, 'Fonts', 'seguiemj.ttf'),
            os.path.join(windir, 'Fonts', 'seguiemj.ttc'),
        ]
    elif system == 'Darwin':
        candidates = ['/System/Library/Fonts/Apple Color Emoji.ttc']
    else:
        candidates = [
            '/usr/share/fonts/truetype/noto/NotoColorEmoji.ttf',
            '/usr/share/fonts/noto-emoji/NotoColorEmoji.ttf',
            '/usr/share/fonts/google-noto-color-emoji/NotoColorEmoji.ttf',
        ]
        candidates += glob.glob('/usr/share/fonts/**/NotoColorEmoji.ttf', recursive=True)
    for path in candidates:
        if os.path.isfile(path):
            return path
    raise FileNotFoundError(
        'No emoji font found. Install one:\n'
        '  Windows:  should have Segoe UI Emoji built-in\n'
        '  macOS:    should have Apple Color Emoji built-in\n'
        '  Linux:    sudo apt install fonts-noto-color-emoji'
    )

_font_path = _find_emoji_font()
_is_noto    = 'Noto' in os.path.basename(_font_path)
_font_size  = 109 if _is_noto else 80
_canvas_sz  = 130 if _is_noto else 100
_emoji_font = ImageFont.truetype(_font_path, _font_size)
print(f'Emoji font: {os.path.basename(_font_path)}')

def emoji_to_image(char, size_px=64):
    """Render one emoji to an RGBA numpy array."""
    canvas = Image.new('RGBA', (_canvas_sz, _canvas_sz), (255, 255, 255, 0))
    draw = ImageDraw.Draw(canvas)
    try:
        draw.text((5, 5), char, font=_emoji_font, embedded_color=True)
    except TypeError:
        draw.text((5, 5), char, font=_emoji_font, fill='black')
    bbox = canvas.getbbox()
    if bbox:
        canvas = canvas.crop(bbox)
    canvas = canvas.resize((size_px, size_px), Image.LANCZOS)
    return np.array(canvas)

print('Setup complete.')

In [ ]:
# ── PARSE CSV ──────────────────────────────────────────
raw = pd.read_csv(CSV_PATH, dtype=str)
raw.columns = [c.strip() for c in raw.columns]
raw = raw.dropna(how='all').reset_index(drop=True)

current_tenant = None
tenant_col = []
for _, row in raw.iterrows():
    tx = str(row.get('TRANSACTION', '')).strip()
    if tx == 'Balance Forward':
        current_tenant = str(row['PROPERTY']).strip()
    tenant_col.append(current_tenant)
raw['TENANT'] = tenant_col

for col in ['DEBIT', 'CREDIT', 'BALANCE']:
    raw[col] = (
        raw[col].astype(str)
        .str.replace(',', '', regex=False)
        .str.replace('"', '', regex=False)
        .str.strip()
    )
    raw[col] = pd.to_numeric(raw[col], errors='coerce')

raw['DATE'] = pd.to_datetime(raw['DATE'], format='%m/%d/%Y', errors='coerce')
raw['MONTH'] = raw['DATE'].dt.to_period('M')
raw['TX'] = raw['TRANSACTION'].astype(str).str.strip()

print(f"Loaded {raw['TENANT'].nunique()} tenants, "
      f"date range {raw['DATE'].min().date()} -> {raw['DATE'].max().date()}")

In [ ]:
# ── END-OF-MONTH BALANCE & FULFILLMENT ────────────────
all_rows = raw.dropna(subset=['TENANT', 'MONTH']).copy()
all_rows['_row'] = range(len(all_rows))

eom = (
    all_rows
    .sort_values(['TENANT', 'DATE', '_row'])
    .groupby(['TENANT', 'MONTH'])
    .last()
    .reset_index()
)[['TENANT', 'MONTH', 'BALANCE']]

pivot = eom.pivot(index='TENANT', columns='MONTH', values='BALANCE').sort_index()

# Monthly fulfillment: % of tenants at $0 or better at month-end
fulfillment_pct = (
    eom.assign(ok=eom['BALANCE'] >= 0)
    .groupby('MONTH')['ok'].mean() * 100
)

print('Data ready.')

In [ ]:
# ── RANDOMIZE EMOJI ASSIGNMENTS ───────────────────────
tenants_sorted = sorted(pivot.index.tolist())
n_tenants = len(tenants_sorted)

pool = EMOJI_POOL[:n_tenants]
assert len(pool) >= n_tenants, (
    f"Need {n_tenants} emojis in EMOJI_POOL, only have {len(pool)}"
)

shuffled = pool.copy()
random.shuffle(shuffled)

id_map = dict(zip(tenants_sorted, shuffled))

# Pre-render all emoji images
emoji_imgs = {e: emoji_to_image(e, size_px=56) for e in shuffled}

# Private key table
key_df = pd.DataFrame([
    {'Emoji': v, 'Tenant': k} for k, v in id_map.items()
]).sort_values('Tenant').reset_index(drop=True)

print(f'Assigned {n_tenants} emojis (re-run to reshuffle).')

In [ ]:
# ── FIGURE: HEATMAP (shareable) ───────────────────────

anon_pivot = pivot.rename(index=id_map)
anon_pivot = anon_pivot.sort_index(key=lambda idx: [shuffled.index(x) for x in idx])
month_labels = [str(m) for m in anon_pivot.columns]
anon_labels  = anon_pivot.index.tolist()

data = anon_pivot.values.astype(float)
n_t = len(anon_labels)
n_m = len(month_labels)

# ── Text colours (all boxes are white) ────────────────
c_txt_prepaid = '#2d6a4f'     # forest green — ahead
c_txt_short   = batlow(0.10)  # dark indigo  — owes
c_txt_zero    = '#999999'     # muted grey   — current

cell_h = 0.7
cell_w = 2.0
fig_w  = cell_w * n_m + 3.0
fig_h  = cell_h * (n_t + 1) + 2.5

fig, ax = plt.subplots(figsize=(fig_w, fig_h))

# ── Draw tenant cells ─────────────────────────────────
for i, emoji_char in enumerate(anon_labels):
    for j in range(n_m):
        val = data[i, j]

        x0 = j * cell_w + 0.05
        y0 = (n_t - 1 - i) * cell_h + 0.05

        rect = mpatches.FancyBboxPatch(
            (x0, y0), cell_w - 0.1, cell_h - 0.1,
            boxstyle='round,pad=0.05',
            facecolor='#FFFFFF', edgecolor='#cccccc', linewidth=1.0
        )
        ax.add_patch(rect)

        if np.isnan(val):
            label, txt_color = '—', '#cccccc'
        elif val > 0:
            label, txt_color = f'+${val:,.0f}', c_txt_prepaid
        elif val < 0:
            label, txt_color = f'-${abs(val):,.0f}', c_txt_short
        else:
            label, txt_color = '$0', c_txt_zero

        ax.text(x0 + (cell_w - 0.1) / 2, y0 + (cell_h - 0.1) / 2,
                label, ha='center', va='center',
                fontsize=10, fontweight='bold', color=txt_color)

# ── Emoji row labels ──────────────────────────────────
for i, emoji_char in enumerate(anon_labels):
    y_center = (n_t - 1 - i) * cell_h + cell_h / 2
    img_arr = emoji_imgs[emoji_char]
    im = OffsetImage(img_arr, zoom=0.45)
    ab = AnnotationBbox(im, (-0.55, y_center),
                        frameon=False, box_alignment=(0.5, 0.5),
                        xycoords='data')
    ax.add_artist(ab)

# ── Summary row: fulfillment % ────────────────────────
summary_y = -cell_h
for j in range(n_m):
    period = anon_pivot.columns[j]
    pct = fulfillment_pct.get(period, 0)

    x0 = j * cell_w + 0.05
    rect = mpatches.FancyBboxPatch(
        (x0, summary_y + 0.05), cell_w - 0.1, cell_h - 0.1,
        boxstyle='round,pad=0.05',
        facecolor='#FFFFFF', edgecolor='#cccccc', linewidth=1.0
    )
    ax.add_patch(rect)

    ax.text(x0 + (cell_w - 0.1) / 2, summary_y + cell_h / 2,
            f'{pct:.0f}%', ha='center', va='center',
            fontsize=11, fontweight='bold', color=c_txt_prepaid)

ax.text(-0.55, summary_y + cell_h / 2, 'Members\nCurrent',
        ha='center', va='center', fontsize=8, fontweight='bold', color='#555555')

# ── Divider ────────────────────────────────────────────
ax.plot([-0.1, n_m * cell_w + 0.1], [0, 0],
        color='#aaaaaa', linewidth=0.8, linestyle='--', clip_on=False)

# ── Month labels on top ────────────────────────────────
for j, month in enumerate(month_labels):
    ax.text(j * cell_w + cell_w / 2, n_t * cell_h + 0.15,
            month, ha='center', va='bottom', fontsize=11, fontweight='bold')

# ── Legend ──────────────────────────────────────────────
legend_elements = [
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c_txt_short,
           markersize=10, label='Short (owes)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c_txt_zero,
           markersize=10, label='Current ($0)'),
    Line2D([0], [0], marker='o', color='w', markerfacecolor=c_txt_prepaid,
           markersize=10, label='Prepaid (ahead)'),
]
ax.legend(handles=legend_elements, loc='lower center',
          bbox_to_anchor=(0.5, -0.02), ncol=3, fontsize=9,
          frameon=False, handlelength=1.5,
          bbox_transform=fig.transFigure)

# ── Title and cleanup ──────────────────────────────────
ax.set_xlim(-1.2, n_m * cell_w + 0.3)
ax.set_ylim(summary_y - 0.4, n_t * cell_h + 0.6)
ax.set_aspect('equal')
ax.axis('off')

fig.suptitle('Monthly Rent Status — HAUS Rosalie',
             fontsize=15, fontweight='bold', y=0.97)
ax.text(n_m * cell_w / 2, n_t * cell_h + 0.8,
        'End-of-month balance per member  ·  Emoji IDs randomized for privacy',
        ha='center', va='bottom', fontsize=9, color='#888888', style='italic')

fig.tight_layout(rect=[0.06, 0.06, 0.98, 0.93])
fig.savefig('rent_heatmap.png', dpi=180, bbox_inches='tight', facecolor='white')
plt.show()
print('\n✓ Heatmap saved to rent_heatmap.png — safe to share.')

In [ ]:
# ── PRIVATE KEY (treasurer eyes only) ─────────────────
print('⚠  PRIVATE — do not share with the group.\n')
print(key_df.to_string(index=False))